# Grad-CAM Explainer Demo

This notebook demonstrates how to use Grad-CAM to explain slide grade predictions.

**What is Grad-CAM?**
- Gradient-weighted Class Activation Mapping
- Highlights which parts of the image the model focused on
- Helps understand WHY the model made a prediction

**How it works:**
1. Model makes a prediction
2. Grad-CAM analyzes which image regions influenced that prediction
3. Creates a heatmap overlay showing important regions

---

## 1. Setup

In [ ]:
import sys
from pathlib import Path
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd

# Add project src to path
project_root = Path('../')
sys.path.insert(0, str(project_root / 'src'))

from models.slide_grade_time_recommender import create_grade_time_model
from data.stain_time_transforms import get_stain_time_val_transforms
from evaluation.gradcam_explainer import GradeExplainer

print("✓ Libraries loaded successfully")

## 2. Load Model

In [ ]:
# Configuration
MODEL_PATH = '../checkpoints_grade_time_balanced/best_model.pth'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device: {DEVICE}")
print(f"Loading model from: {MODEL_PATH}")

# Load model (dual-head: grade + time)
model = create_grade_time_model(
    architecture='resnet18',
    num_grade_classes=5,
    pretrained=False,
    use_time_context=True
)
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("✓ Model loaded successfully (dual-head: grade + time)")

# Create explainer
explainer = GradeExplainer(model, device=DEVICE)
print("✓ Grad-CAM explainer initialized")

## 3. Load Test Images

In [ ]:
# Load test split metadata
test_df = pd.read_csv('../data/data_04/splits/test.csv')

print(f"Test set size: {len(test_df)} images")
print(f"\nGrade distribution:")
print(test_df['grade_numeric'].value_counts().sort_index())

# Select sample images (one from each grade)
sample_images = []
for grade in sorted(test_df['grade_numeric'].unique()):
    sample = test_df[test_df['grade_numeric'] == grade].iloc[0]
    sample_images.append(sample)

print(f"\n✓ Selected {len(sample_images)} sample images (one per grade)")

## 4. Generate Grad-CAM Explanations

In [ ]:
# Image preprocessing transform
transform = get_stain_time_val_transforms((512, 512))

# Process each sample
explanations = []

for idx, sample in enumerate(sample_images):
    print(f"\nProcessing image {idx+1}/{len(sample_images)}: {sample['filename']}")
    
    # Load image
    image_path = Path(sample['filepath'] if 'filepath' in sample else sample['filename'])
    image = Image.open(image_path).convert('RGB')
    image_np = np.array(image)
    
    # Preprocess
    image_tensor = transform(image).unsqueeze(0)
    
    # Generate explanation
    explanation = explainer.explain(image_tensor, image_np)
    
    # Store results
    explanations.append({
        'sample': sample,
        'image': image_np,
        'explanation': explanation
    })
    
    print(f"  True Grade: {sample['grade_numeric']}")
    print(f"  Predicted Grade: {explanation['grade_numeric']}")
    print(f"  Status: {explanation['status']}")
    print(f"  Confidence: {explanation['confidence']:.2%}")
    print(f"  Reason: {explanation['reason']}")

print("\n✓ All explanations generated")

## 5. Visualize Results

In [ ]:
# Create visualization for each sample
for idx, result in enumerate(explanations):
    sample = result['sample']
    image_np = result['image']
    explanation = result['explanation']
    
    # Create figure
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Original image
    axes[0].imshow(image_np)
    axes[0].set_title(
        f"Original Image\nTrue Grade: {sample['grade_numeric']}",
        fontsize=12, fontweight='bold'
    )
    axes[0].axis('off')
    
    # Grad-CAM heatmap
    im = axes[1].imshow(explanation['cam_heatmap'], cmap='jet')
    axes[1].set_title('Grad-CAM Heatmap\n(Model Attention)', fontsize=12, fontweight='bold')
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    
    # Overlay
    axes[2].imshow(explanation['overlay_image'])
    status_color = 'green' if explanation['status'] == 'PASS' else 'red'
    axes[2].set_title(
        f"Predicted: Grade {explanation['grade_label']} ({explanation['status']})\n"
        f"{explanation['reason']}\n"
        f"Confidence: {explanation['confidence']:.2%}",
        fontsize=12, fontweight='bold', color=status_color
    )
    axes[2].axis('off')
    
    plt.suptitle(
        f"Sample {idx+1}: {sample['dilution']} dilution, {sample['time_minutes']} min, {sample['smear_type']} smear",
        fontsize=14, fontweight='bold', y=1.02
    )
    
    plt.tight_layout()
    plt.show()
    
    # Print probability breakdown
    print(f"\nSample {idx+1} - Probability Breakdown:")
    for i, prob in enumerate(explanation['probabilities']):
        grade_label = ['I', 'II', 'III', 'IV', 'V'][i]
        bar = '█' * int(prob * 50)
        print(f"  Grade {grade_label}: {prob:6.2%} {bar}")
    print()

## 6. Test on Custom Image (Optional)

In [ ]:
# Specify your custom image path here
CUSTOM_IMAGE_PATH = '../data/data_04/processed/10%_15min_3_thin.jpg'

# Load and process
if Path(CUSTOM_IMAGE_PATH).exists():
    print(f"Loading custom image: {CUSTOM_IMAGE_PATH}")
    
    image = Image.open(CUSTOM_IMAGE_PATH).convert('RGB')
    image_np = np.array(image)
    image_tensor = transform(image).unsqueeze(0)
    
    # Generate explanation
    explanation = explainer.explain(image_tensor, image_np)
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    axes[0].imshow(image_np)
    axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(explanation['cam_heatmap'], cmap='jet')
    axes[1].set_title('Grad-CAM Heatmap', fontsize=14, fontweight='bold')
    axes[1].axis('off')
    
    axes[2].imshow(explanation['overlay_image'])
    axes[2].set_title(
        f"Grade {explanation['grade_label']}: {explanation['reason']}\n"
        f"Confidence: {explanation['confidence']:.2%}",
        fontsize=14, fontweight='bold',
        color='green' if explanation['status'] == 'PASS' else 'red'
    )
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nPrediction Results:")
    print(f"  Grade: {explanation['grade_label']} ({explanation['grade_numeric']})")
    print(f"  Status: {explanation['status']}")
    print(f"  Confidence: {explanation['confidence']:.2%}")
    print(f"  Reason: {explanation['reason']}")
else:
    print(f"Image not found: {CUSTOM_IMAGE_PATH}")
    print("Please update CUSTOM_IMAGE_PATH to a valid image path")

---

## Summary

This notebook demonstrated how to use Grad-CAM to explain slide grade predictions:

1. **Visual Explanations**: Heatmaps show which image regions influenced the prediction
2. **Text Explanations**: Simple reason text based on predicted grade
3. **Confidence Scores**: Probability distributions across all grades

### Key Insights:

- **Red/Hot regions**: Areas the model focused on most
- **Blue/Cool regions**: Areas the model ignored
- **For failed slides**: Heatmap often highlights problematic staining regions
- **For passed slides**: Heatmap shows well-stained cellular regions

### Next Steps:

1. **API Integration**: Use `/api/explain-grade` endpoint in your frontend
2. **User Feedback**: Show Grad-CAM overlays to users
3. **Quality Control**: Use explanations to identify systematic issues

---